<a href="https://colab.research.google.com/github/anupseth360-netizen/python-lab/blob/main/LAB_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IT12

In [ ]:
import os

# Ensure the data directory exists
os.makedirs("data", exist_ok=True)

log_path = "data/shift_log.txt"

# 15 records including 3 deliberately filthy/malformed ones
log_records = [
    "2026-11-06, SHIFT-A, 105, OK",
    "2026-11-06, SHIFT-A, 110, OK",
    "INVALID_LINE_NO_COMMA_OR_DATA",
    "2026-11-06, SHIFT-A, 98, OK",
    "2026-11-06, SHIFT-B, 150, OK",
    "2026-11-06, SHIFT-B, -50, ERROR",
    "2026-11-06, SHIFT-B, 142, OK",
    "2026-11-06, SHIFT-C, 200, OK",
    "CORRUPT|DATA|FORMAT|HERE",
    "2026-11-06, SHIFT-C, 210, OK",
    "2026-11-06, SHIFT-C, 195, OK",
    "2026-11-06, SHIFT-A, 115, OK",
    "ANOTHER_BAD_ENTRY_999",
    "2026-11-06, SHIFT-B, 130, OK",
    "2026-11-06, SHIFT-C, 180, OK",
]

# Write records to file
with open(log_path, "w") as f:
    for record in log_records:
        f.write(record + "\n")

# Read back and print line count
with open(log_path, "r") as f:
    lines = f.readlines()
    print(f"Total lines in {log_path}: {len(lines)}")

Total lines in data/shift_log.txt: 15


IT10

In [ ]:
import csv
import os

os.makedirs("data", exist_ok=True)

input_file = "data/shift_log.txt"
report_file = "data/report.txt"
rejects_file = "data/rejects.csv"

valid_records = []
rejects = []

# Read & Parse & Validate
if os.path.exists(input_file):
    with open(input_file, "r") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            parts = [p.strip() for p in line.split(",")]

            # Validation rule: must have exactly 4 parts
            if len(parts) != 4:
                rejects.append([line_num, line, "Incorrect number of fields"])
                continue

            date, shift, val_str, status = parts
            try:
                val = float(val_str)
                if val < 0:
                    raise ValueError("Negative value not allowed")
            except ValueError as e:
                rejects.append([line_num, line, str(e)])
                continue

            valid_records.append({"date": date, "shift": shift, "value": val, "status": status})

# Aggregate (e.g., sum values per shift)
shift_summary = {}
for rec in valid_records:
    s = rec["shift"]
    shift_summary[s] = shift_summary.get(s, 0.0) + rec["value"]

# Write Report
with open(report_file, "w") as f:
    f.write(f"{'Shift':<10} | {'Total Value':<12}\n")
    f.write("-" * 25 + "\n")
    for shift, total in shift_summary.items():
        f.write(f"{shift:<10} | {total:<12.2f}\n")

# Write Rejects CSV
with open(rejects_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["LineNumber", "RawLine", "Reason"])
    writer.writerows(rejects)

print("Pipeline executed successfully. Reports generated.")

Pipeline executed successfully. Reports generated.


IT8

In [ ]:
import re

# Example line containing a part code pattern (e.g., ABC-1234)
test_lines = [
    "Item: XYZ-9876, Qty: 10",
    "Item: bad_format, Qty: 5",
    "Item: AB-123, Qty: 2" # Invalid code shape
]

# Regex pattern tolerates spacing around separators and validates shape (3 letters, hyphen, 4 digits)
pattern = re.compile(r"Item:\s*([A-Z]{3}-\d{4})\s*,\s*Qty:\s*(\d+)")

for line in test_lines:
    # --- OLD VERSION (Commented out) ---
    # parts = line.split(",")
    # print(parts)

    # --- NEW REGEX VERSION ---
    match = pattern.search(line)
    if match:
        part_code, qty = match.groups()
        print(f"Matched! Code: {part_code}, Quantity: {qty}")
    else:
        print(f"Rejected line: {line}")

Matched! Code: XYZ-9876, Quantity: 10
Rejected line: Item: bad_format, Qty: 5
Rejected line: Item: AB-123, Qty: 2


IT6

In [ ]:
import csv
import datetime
import glob
import os

os.makedirs("data", exist_ok=True)

history_file = "data/run_history.csv"
timestamp = datetime.datetime.now().isoformat()

# Ensure history file has headers if new
if not os.path.exists(history_file):
    with open(history_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Timestamp", "FileName", "RecordsProcessed", "RejectsCount"])

# Process a whole folder of .txt files
txt_files = glob.glob("data/*.txt")

for filepath in txt_files:
    filename = os.path.basename(filepath)
    if filename == "report.txt": # Skip output reports if they are in data/
        continue

    records_processed = 0
    rejects_count = 0

    with open(filepath, "r") as f:
        for line in f:
            if line.strip():
                records_processed += 1
                if "," not in line: # Simple check mimicking error
                    rejects_count += 1

    # Generate individual plant summary report
    summary_filename = f"data/summary_{filename}"
    with open(summary_filename, "w") as summary_f:
        summary_f.write(f"Summary for {filename}\n")
        summary_f.write(f"Processed: {records_processed}, Rejects: {rejects_count}\n")

    # Append run history
    with open(history_file, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([timestamp, filename, records_processed, rejects_count])

print("Batch job completed for all text files.")

Batch job completed for all text files.
